# Reasoning Distillation: GRPO + Eval + Publish

**Runtime:** Colab Pro H100 (80GB VRAM)
**Secrets:** `TINKER_API_KEY`, `HF_TOKEN`

1. Install + setup
2. Download SFT LoRA weights from Tinker
3. GRPO reinforcement learning
4. Merge all models
5. lm-eval benchmarks (industry standard)
6. Trick questions side-by-side
7. Push best model to HuggingFace + GGUF

In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. INSTALL — restart runtime after this cell!
# ═══════════════════════════════════════════════════════════
# Uses Unsloth's official installer (handles transformers v5 for Qwen3.5)
# Standard pip install does NOT work — pins transformers<=4.57.6
!curl -fsSL https://unsloth.ai/install.sh | sh
!pip install tinker lm-eval --quiet
print('\n✅ Done. Now: Runtime → Restart session, then run the NEXT cell.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 2. SETUP — run after restart, then Run All from here down
# ═══════════════════════════════════════════════════════════
import transformers
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
assert 'qwen3_5' in CONFIG_MAPPING, f'qwen3_5 not in transformers {transformers.__version__}! Re-run cell 1 and restart.'
print(f'transformers {transformers.__version__} — qwen3_5 ✅')

from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, glob, subprocess, re
DRIVE = '/content/drive/MyDrive/distillreasoning'
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(f'{DRIVE}/eval_results', exist_ok=True)
BASE_MODEL = 'Qwen/Qwen3.5-4B'
print(f'Drive: {DRIVE} ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 3. DOWNLOAD SFT LoRA weights from Tinker
# ═══════════════════════════════════════════════════════════
import tinker, requests, tarfile

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')
service = tinker.ServiceClient()
rest = service.create_rest_client()

SFT_CHECKPOINTS = {
    'glm5': 'tinker://0fbca836-2aae-5500-b28d-93c2a46a328b:train:0/sampler_weights/qwen35-4b-glm5-final',
    'kimi': 'tinker://f5795e66-71e4-5cf4-9ebe-1cc14c27aa6e:train:0/sampler_weights/qwen35-4b-kimi-final',
    'combined': 'tinker://41e7bd9e-e49f-5f13-a5ff-f4339faab448:train:0/sampler_weights/qwen35-4b-combined-final',
}

for name, tinker_path in SFT_CHECKPOINTS.items():
    lora_dir = f'{DRIVE}/sft_lora_{name}'
    if os.path.exists(lora_dir) and any(f.endswith('.safetensors') for f in os.listdir(lora_dir)):
        print(f'{name}: already downloaded')
    else:
        print(f'Downloading {name}...')
        url_resp = rest.get_checkpoint_archive_url_from_tinker_path(tinker_path).result()
        local_tar = f'/tmp/lora_{name}.tar'
        r = requests.get(url_resp.url, stream=True)
        with open(local_tar, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): f.write(chunk)
        os.makedirs(lora_dir, exist_ok=True)
        with tarfile.open(local_tar) as t: t.extractall(lora_dir)
        print(f'  Downloaded: {os.listdir(lora_dir)}')

    # Patch adapter_config — Tinker leaves base_model_name_or_path as null
    config_path = os.path.join(lora_dir, 'adapter_config.json')
    if os.path.exists(config_path):
        config = json.load(open(config_path))
        if config.get('base_model_name_or_path') is None:
            config['base_model_name_or_path'] = BASE_MODEL
            json.dump(config, open(config_path, 'w'), indent=2)
            print(f'  Patched adapter_config ✅')

print('\nAll LoRA weights ready! ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 4. GRPO SETUP — dataset + reward functions
# ═══════════════════════════════════════════════════════════
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from datasets import load_dataset
import torch

max_seq_length = 2048
lora_rank = 32

SYSTEM_PROMPT = """Respond in the following format:
<reasoning>\n...\n</reasoning>\n<answer>\n...\n</answer>"""

def extract_hash_answer(text):
    return text.split('####')[1].strip() if '####' in text else None

dataset = load_dataset('openai/gsm8k', 'main')['train'].map(lambda x: {
    'prompt': [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': x['question']}],
    'answer': extract_hash_answer(x['answer']),
})
print(f'GRPO dataset: {len(dataset)} problems ✅')

def extract_xml_answer(text):
    if '<answer>' in text and '</answer>' in text:
        return text.split('<answer>')[-1].split('</answer>')[0].strip()
    nums = re.findall(r'-?\d+\.?\d*', text.replace(',', ''))
    return nums[-1] if nums else ''

def correctness_reward(prompts, completions, answer, **kwargs):
    return [2.0 if extract_xml_answer(c[0]['content']) == a else 0.0 for c, a in zip(completions, answer)]

def format_reward(completions, **kwargs):
    return [0.5 if '<reasoning>' in c[0]['content'] and '<answer>' in c[0]['content'] else 0.0 for c in completions]

def int_reward(completions, **kwargs):
    return [0.5 if extract_xml_answer(c[0]['content']).replace('-','').replace('.','').isdigit() else 0.0 for c in completions]

print('Reward functions ready ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 5. RUN GRPO — saves each model to Drive
# ═══════════════════════════════════════════════════════════
for teacher in ['kimi', 'glm5', 'combined']:
    grpo_dir = f'{DRIVE}/grpo_lora_{teacher}'
    if os.path.exists(grpo_dir) and os.listdir(grpo_dir):
        print(f'{teacher}: GRPO already done ✅')
        continue

    print(f'\n{"="*50}\nGRPO: {teacher}\n{"="*50}')
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=f'{DRIVE}/sft_lora_{teacher}',
        max_seq_length=max_seq_length, load_in_4bit=True,
        fast_inference=True, max_lora_rank=lora_rank, gpu_memory_utilization=0.6,
    )
    trainer = GRPOTrainer(
        model=model, processing_class=tokenizer,
        reward_funcs=[correctness_reward, format_reward, int_reward],
        args=GRPOConfig(
            learning_rate=5e-6, adam_beta1=0.9, adam_beta2=0.99, weight_decay=0.1,
            warmup_ratio=0.1, lr_scheduler_type='cosine', optim='paged_adamw_8bit',
            logging_steps=1, per_device_train_batch_size=1, gradient_accumulation_steps=4,
            num_generations=8, max_prompt_length=512, max_completion_length=max_seq_length-512,
            max_steps=250, save_steps=250, max_grad_norm=0.1,
            report_to='none', output_dir=f'/tmp/grpo_{teacher}',
        ),
        train_dataset=dataset,
    )
    trainer.train()
    model.save_lora(grpo_dir)
    print(f'  ✅ Saved: {grpo_dir}')
    del model, tokenizer, trainer; torch.cuda.empty_cache()

print('\nAll GRPO training complete! ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 6. MERGE all LoRA adapters with base model
# ═══════════════════════════════════════════════════════════
lora_dirs = {}
for prefix in ['sft_lora', 'grpo_lora']:
    for teacher in ['glm5', 'kimi', 'combined']:
        d = f'{DRIVE}/{prefix}_{teacher}'
        if os.path.exists(d) and any(f.endswith('.safetensors') for f in os.listdir(d)):
            lora_dirs[f'{prefix.replace("_lora","")}_{teacher}'] = d

for name, lora_dir in lora_dirs.items():
    merged = f'{DRIVE}/merged_{name}'
    if os.path.exists(merged) and any(f.endswith('.safetensors') for f in os.listdir(merged)):
        print(f'{name}: already merged ✅')
        continue
    print(f'Merging {name}...')
    model, tok = FastLanguageModel.from_pretrained(model_name=lora_dir, max_seq_length=4096, load_in_4bit=False)
    model.save_pretrained_merged(merged, tok, save_method='merged_16bit')
    del model, tok; torch.cuda.empty_cache()
    print(f'  ✅ {merged}')

print('\nAll models merged! ✅')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 7. LM-EVAL — industry standard benchmarks
# ═══════════════════════════════════════════════════════════
# ARC/GPQA/MMLU use log-likelihood (no extraction bugs)
# GSM8K/MATH use generation with proper extraction

ALL_TASKS = 'gsm8k_cot,minerva_math,arc_challenge,gpqa_diamond,mmlu_pro'

EVAL_MODELS = [
    ('base-qwen35-4b', 'Qwen/Qwen3.5-4B', True),
    ('llama-3.2-3b', 'meta-llama/Llama-3.2-3B', False),
    ('qwen3-8b', 'Qwen/Qwen3-8B', False),
]
for name in lora_dirs:
    merged = f'{DRIVE}/merged_{name}'
    if os.path.exists(merged): EVAL_MODELS.append((name, merged, True))

print(f'{len(EVAL_MODELS)} models to evaluate')

for name, path, is_chat in EVAL_MODELS:
    rd = f'{DRIVE}/eval_results/{name}'
    if glob.glob(f'{rd}/**/results.json', recursive=True):
        print(f'{name}: done ✅')
        continue
    print(f'Evaluating {name}...')
    cmd = ['lm-eval', '--model', 'hf',
           '--model_args', f'pretrained={path},dtype=bfloat16,trust_remote_code=True',
           '--tasks', ALL_TASKS, '--batch_size', 'auto',
           '--output_path', rd, '--log_samples']
    if is_chat: cmd.extend(['--apply_chat_template', '--fewshot_as_multiturn'])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"}')
    print(r.stdout[-500:])

In [ ]:
# ═══════════════════════════════════════════════════════════
# 8. RESULTS TABLE
# ═══════════════════════════════════════════════════════════
all_results = {}
for name, _, _ in EVAL_MODELS:
    files = glob.glob(f'{DRIVE}/eval_results/{name}/**/results.json', recursive=True)
    if not files: continue
    data = json.load(open(files[0]))
    r = {}
    for task, td in data.get('results', {}).items():
        for m in ['exact_match,flexible-extract','exact_match,strict-match','acc_norm,none','acc,none']:
            if m in td: r[task] = round(td[m]*100, 1); break
    all_results[name] = r

benches = ['gsm8k_cot','minerva_math','arc_challenge','gpqa_diamond','mmlu_pro']
labels = ['GSM8K','MATH','ARC','GPQA','MMLU-Pro']
print(f'{"Model":25s}' + ''.join(f'{l:>10s}' for l in labels))
print('-'*75)
for name, _, _ in EVAL_MODELS:
    if name not in all_results: continue
    print(f'{name:25s}' + ''.join(f'{str(all_results[name].get(b,"—"))+"%":>10s}' for b in benches))

json.dump(all_results, open(f'{DRIVE}/eval_results/comparison.json','w'), indent=2)
print(f'\nSaved: {DRIVE}/eval_results/comparison.json')

In [ ]:
# ═══════════════════════════════════════════════════════════
# 9. TRICK QUESTIONS — side by side
# ═══════════════════════════════════════════════════════════
TRICKS = [
    ('A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?', '9'),
    ('If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?', '5 minutes'),
    ('A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?', '$0.05'),
]
SYS = 'You are a helpful reasoning assistant. Think through problems step by step.'

for mn in ['base-qwen35-4b', 'sft_kimi', 'grpo_kimi']:
    path = next((p for n,p,_ in EVAL_MODELS if n==mn), None)
    if not path: print(f'{mn}: not found'); continue
    model, tok = FastLanguageModel.from_pretrained(model_name=path, max_seq_length=2048, load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    print(f'\n=== {mn} ===')
    for q, exp in TRICKS:
        inp = tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':q}], return_tensors='pt', add_generation_prompt=True).to('cuda')
        out = model.generate(inp, max_new_tokens=512, temperature=0.6, do_sample=True)
        print(f'  Q: {q[:60]}...\n  Expected: {exp}\n  Model: {tok.decode(out[0][inp.shape[-1]:], skip_special_tokens=True)[:200]}\n')
    del model, tok; torch.cuda.empty_cache()

In [ ]:
# ═══════════════════════════════════════════════════════════
# 10. PUBLISH — push best model to HuggingFace + GGUF
# ═══════════════════════════════════════════════════════════
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

our = {k:v for k,v in all_results.items() if 'sft' in k or 'grpo' in k}
best = max(our, key=lambda k: our[k].get('gsm8k_cot', 0))
print(f'Best: {best} — {our[best]}')

HF_REPO = 'bmeyer2025/qwen3.5-4b-reasoning-distilled'
model, tok = FastLanguageModel.from_pretrained(model_name=f'{DRIVE}/merged_{best}', max_seq_length=4096, load_in_4bit=False)

model.push_to_hub_merged(HF_REPO, tok, save_method='merged_16bit', token=userdata.get('HF_TOKEN'))
print(f'Merged model pushed ✅')

model.push_to_hub_gguf(HF_REPO, tok, quantization_method=['q4_k_m','q8_0'], token=userdata.get('HF_TOKEN'))
print(f'GGUF pushed ✅')
print(f'\nRun: ollama run hf.co/{HF_REPO}:Q4_K_M')